<a href="https://colab.research.google.com/github/thisisaadi123/chronos-project/blob/main/Chronos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multivariate Retail Forecasting via Foundation Models
## Causal Inference on Intermittent Demand Sequences

### Abstract
Forecasting intermittent retail demand presents a significant challenge for traditional statistical frameworks due to high sparsity and non-stationary variance. This project implements a zero-shot forecasting pipeline leveraging the Chronos-T5 transformer architecture. By aligning target sales sequences with dynamic pricing and temporal covariates, the model generates probabilistic risk envelopes (P10–P90) used for inventory optimization.

### Technical Implementation
* Data Engineering: Horizontal-to-vertical unpivoting (Melting) and multi-source feature alignment.
* Normalization: Localized standardization followed by Inverse Hyperbolic Sine (arcsinh) scaling to handle 0-value density and promotional outliers.
* Architecture: 3D Tensor formulation for Alternating Intra/Inter-series Attention.
* Evaluation: Calibration testing via Mean Weighted Quantile Loss (wQL) to assess probability envelope accuracy.

### Raw Data Architecture (Data Dictionary)

Before manipulating the data, we must understand the schema of the three core tables provided by Walmart. To build a multivariate causal model, we will need to extract and join specific columns of interest across these tables.

#### 1. Sales Data (`sales_raw`)
This table records the daily unit sales per product. It is formatted as a "Wide" dataframe.
* `id` *(string)*: Unique identifier combining the item and store. **[Target Key]**
* `item_id`, `dept_id`, `cat_id` *(string)*: Product hierarchy metadata.
* `store_id`, `state_id` *(string)*: Geographical metadata.
* `d_1` to `d_1941` *(integer)*: The exact number of units sold on day $d$. **[Primary Target Variable]**

#### 2. Calendar Data (`calendar_raw`)
This table maps the day indices (`d_1`) to real-world dates, weeks, and special events.
* `date` *(string)*: Standard YYYY-MM-DD format.
* `wm_yr_wk` *(integer)*: Walmart's proprietary weekly ID. **[Join Key for Pricing]**
* `weekday`, `wday`, `month`, `year`: Standard temporal features.
* `d` *(string)*: The day index (e.g., "d_1") matching the sales columns. **[Join Key for Sales]**
* `event_name_1`, `event_type_1` *(string/NaN)*: Cultural/national holidays (e.g., Super Bowl, Christmas). **[Covariate of Interest]**
* `snap_CA`, `snap_TX`, `snap_WI` *(binary)*: Indicates if SNAP (food stamps) purchases were allowed that day.

#### 3. Pricing Data (`prices_raw`)
This table logs the historical weekly price of every item in every store.
* `store_id` *(string)*: Store location. **[Join Key]**
* `item_id` *(string)*: The specific product. **[Join Key]**
* `wm_yr_wk` *(integer)*: Walmart's weekly ID. **[Join Key]**
* `sell_price` *(float)*: The retail price of the item for that specific week. **[Covariate of Interest]**

**Objective:** We must isolate a specific `id`'s sales history, melt it into a vertical sequence, and map the `event_name_1` and `sell_price` to that timeline to create our final `[Batch, Variates, Length]` tensor.

In [ ]:
import os
import pandas as pd

print("[INFO] Authenticating and downloading M5 dataset...")
os.environ['KAGGLE_API_TOKEN'] = "KGAT_a66b802dffecdfd4512d0a850e39d710"
!kaggle competitions download -c m5-forecasting-accuracy
!unzip -o -q m5-forecasting-accuracy.zip

print("[INFO] Loading raw dataframes into memory...")
# We load a small subset of sales to save RAM, but the full calendar and pricing histories
sales_raw = pd.read_csv('sales_train_evaluation.csv', nrows=50)
calendar_raw = pd.read_csv('calendar.csv')
prices_raw = pd.read_csv('sell_prices.csv')

print("[SUCCESS] Data loaded successfully. Ready for inspection.")

[INFO] Authenticating and downloading M5 dataset...
m5-forecasting-accuracy.zip: Skipping, found more recently modified local copy (use --force to force download)
[INFO] Loading raw dataframes into memory...
[SUCCESS] Data loaded successfully. Ready for inspection.


### Phase 1: Sequence Formatting (The "Melt")

**The Concept of "Melting":**

In data science, datasets often arrive in a "Wide" format where time expands horizontally (e.g., Day 1, Day 2, and Day 3 are separate columns). While this is easy for humans to read in a spreadsheet, it breaks time-series AI models. Neural networks cannot read left-to-right across columns; they require a continuous, top-to-bottom sequence.

"Melting" (or unpivoting) is the programmatic process of crushing those horizontal columns into a "Long" format. We take all 1,941 day columns and compress them into just two clean, vertical columns: `day` and `sales`.

**Our Objective:**
We will use the pandas `melt` function to transform the raw Walmart data into a sequential timeline. Once the data flows vertically, we will isolate a single Hobby item to serve as the primary target variable for our foundation model.
### Phase 1: Executing the "Melt" (Target Extraction)

Now that we understand the Wide format, we will use pandas to crush the 1,941 day columns into a vertical timeline.

Once the data is vertical, we will isolate a single product (our "Target Item") to serve as the baseline for our foundation model. We will print the dataframe before and after this process so the transformation is clearly visible.

In [ ]:
# --- 1. THE MELT TRANSFORMATION ---
# We define the "ID" columns. These are the columns we DO NOT want to crush.
id_vars = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']

# The melt function takes all the other columns (d_1 to d_1941) and stacks them vertically.
# 'var_name' becomes the new column holding "d_1", and 'value_name' holds the actual sales number.
sales_long = sales_raw.melt(id_vars=id_vars, var_name='day_string', value_name='sales')

# --- 2. CLEANING THE TIMELINE ---
# "d_1" is a string. AI models prefer pure numbers.
# We split the text at the underscore '_' and keep the second part (the integer).
sales_long['day'] = sales_long['day_string'].str.replace('d_', '').astype(int)

# We sort the data so Day 1 comes before Day 2, and drop the old string column to save memory.
sales_long = sales_long.sort_values(['id', 'day']).drop(columns=['day_string'])

# --- 3. ISOLATING THE TARGET ---
# We grab the ID of the very first item in our dataset.
target_item = sales_long['id'].iloc[0]

# We filter our massive vertical dataframe to ONLY include this specific item's history.
item_story = sales_long[sales_long['id'] == target_item].copy()

print(f"[SUCCESS] Target Extracted: {target_item}")
print("Notice how time now flows downward in the 'day' column:")
display(item_story[['id', 'day', 'sales']].head(5))

[SUCCESS] Target Extracted: HOBBIES_1_001_CA_1_evaluation
Notice how time now flows downward in the 'day' column:


,id,day,sales
0,HOBBIES_1_001_CA_1_evaluation,1,0
50,HOBBIES_1_001_CA_1_evaluation,2,0
100,HOBBIES_1_001_CA_1_evaluation,3,0
150,HOBBIES_1_001_CA_1_evaluation,4,0
200,HOBBIES_1_001_CA_1_evaluation,5,0


### Phase 2: Engineering Causal Covariates

Time-series forecasting without covariates is just guessing. To predict *when* a spike will occur, the model needs to know why historical spikes occurred.

We will engineer two external variables (Covariates):
1. **Events (Temporal Covariate):** We will convert the Calendar's text events (e.g., "Thanksgiving") into a binary flag (1 for event, 0 for normal day).
2. **Prices (Dynamic Covariate):** We will map the weekly price of the item onto our daily timeline.

In [ ]:
# --- 1. PROCESSING THE CALENDAR ---
# We slice only the columns we need from the raw calendar to save RAM.
# We also clean the 'd_1' string into an integer so it matches our sales data perfectly.
calendar_subset = calendar_raw[['d', 'event_name_1', 'wm_yr_wk']].copy()
calendar_subset['day'] = calendar_subset['d'].str.replace('d_', '').astype(int)

# We create a binary flag. If 'event_name_1' is NOT empty (notna), it becomes a 1. Otherwise, 0.
calendar_subset['event_flag'] = calendar_subset['event_name_1'].notna().astype(int)

# --- 2. MERGING EVENTS INTO SALES ---
# We join the calendar data onto our item's sales timeline.
# We match them up using the 'day' column (e.g., Day 45 in Sales connects to Day 45 in Calendar).
item_df = pd.merge(item_story, calendar_subset[['day', 'wm_yr_wk', 'event_flag']], on='day', how='left')

# --- 3. MERGING PRICES INTO SALES ---
# We filter the massive price dataset to find ONLY the prices for our specific item in its specific store.
item_prices = prices_raw[(prices_raw['item_id'] == item_df['item_id'].iloc[0]) &
                         (prices_raw['store_id'] == item_df['store_id'].iloc[0])]

# We merge the price onto the sales timeline using the Walmart Weekly ID ('wm_yr_wk').
item_df = pd.merge(item_df, item_prices[['wm_yr_wk', 'sell_price']], on='wm_yr_wk', how='left')

# --- 4. HANDLING MISSING DATA (NaNs) ---
# If the item wasn't sold on Day 1, its price will be NaN (Not a Number).
# bfill() copies the first known price backward. ffill() copies the last known price forward.
item_df['sell_price'] = item_df['sell_price'].bfill().ffill()

print("[SUCCESS] Covariates Engineered.")
print("The AI can now see the Price and Event status for every single day of sales:")
display(item_df[['day', 'sales', 'event_flag', 'sell_price']].tail(5))

[SUCCESS] Covariates Engineered.
The AI can now see the Price and Event status for every single day of sales:


,day,sales,event_flag,sell_price
1936,1937,0,0,8.38
1937,1938,3,0,8.38
1938,1939,3,0,8.38
1939,1940,0,0,8.38
1940,1941,1,0,8.38


### Phase 3: Visualizing the "Causal Tangle"

Before applying AI, we must visually verify the problem we are trying to solve. We will plot our Target (Sales) against our Covariate (Price).

Observe the extreme sparsity—days of zero sales followed by sudden spikes. More importantly, observe the pricing behavior: if the price suddenly drops or increases, does it alter the frequency of the sales spikes? This complex relationship is exactly what Chronos-T5 is designed to decode.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- 1. CHART SETUP ---
# We create a chart with two Y-axes.
# The left axis will be for Sales count, the right axis for Price in dollars.
fig = make_subplots(specs=[[{"secondary_y": True}]])
plot_data = item_df.tail(200) # We only plot the last 200 days so the spikes are visible.

# --- 2. PLOTTING THE DATA ---
# We plot the daily sales as a solid black line on the primary (left) axis.
fig.add_trace(go.Scatter(x=plot_data['day'], y=plot_data['sales'],
                         name="Daily Sales", line=dict(color='#111')), secondary_y=False)

# We plot the sell price as a dotted blue line on the secondary (right) axis.
fig.add_trace(go.Scatter(x=plot_data['day'], y=plot_data['sell_price'],
                         name="Sell Price ($)", line=dict(color='#0066FF', dash='dot')), secondary_y=True)

# --- 3. RENDERING ---
# We apply a clean, professional theme and display the interactive chart.
fig.update_layout(title="Retail Causality: Price vs. Demand", template="plotly_white", height=400)
fig.show()

### Phase 4: Robust Scaling & Tensor Formatting

**The Math ($\sinh^{-1}$):**
Foundation models struggle with extreme variance. If a product usually sells 0 units but spikes to 20 during an event, that sudden "20" acts like a megaphone, blinding the model's attention mechanism. To fix this, we apply an **Inverse Hyperbolic Sine ($\sinh^{-1}$)** transformation. This elegantly squashes extreme outliers while safely handling the zeros (unlike a standard logarithm, which breaks on zeros).

**The Architecture:**
Chronos requires inputs in a highly specific mathematical shape: a 3-Dimensional PyTorch Tensor structured as `[Batch, Variates, Sequence Length]`.

In [ ]:
import torch
import numpy as np

# --- 1. THE ROBUST SCALER FUNCTION ---
def chronos_robust_scale(series):
    # Calculate the average (mu) and spread (sigma) of the data. Add a tiny number to prevent divide-by-zero.
    mu, sigma = series.mean(), series.std() + 1e-8

    # Center the data around zero, then apply the Inverse Hyperbolic Sine squash.
    z_score = (series - mu) / sigma
    squashed = np.arcsinh(z_score)
    return squashed, mu, sigma

# --- 2. APPLYING THE SCALER ---
# We scale all three of our data columns uniformly so they speak the same mathematical "language".
y_scaled, y_mu, y_sigma = chronos_robust_scale(item_df['sales'])
p_scaled, p_mu, p_sigma = chronos_robust_scale(item_df['sell_price'])
e_scaled, e_mu, e_sigma = chronos_robust_scale(item_df['event_flag'])

# --- 3. DEFINING THE HORIZON ---
# We feed the model the last 512 days of history to predict the next 28 days of the future.
CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 28

# --- 4. TENSOR CONSTRUCTION ---
# We slice the last 512 days of data and convert them into PyTorch Tensors (the format Deep Learning models use).
context_y = torch.tensor(y_scaled.values[-CONTEXT_LENGTH:]).float()
context_p = torch.tensor(p_scaled.values[-CONTEXT_LENGTH:]).float()
context_e = torch.tensor(e_scaled.values[-CONTEXT_LENGTH:]).float()

# We stack the three lines on top of each other into a single 2D grid: [3 Variates, 512 Days]
context_tensor = torch.stack([context_y, context_p, context_e])
print(f"[INFO] Mathematical Tensor ready. Shape: {context_tensor.shape}")

[INFO] Mathematical Tensor ready. Shape: torch.Size([3, 512])


### Phase 5: Zero-Shot Chronos Inference

We initialize the `amazon/chronos-t5-base` pipeline. Because Chronos is a Foundation Model, we **do not** train it. Instead, we control its performance by tuning its generation hyperparameters:
* `num_samples=50`: Generates 50 distinct future paths to build a robust probability curve.
* `temperature=0.8`: Reduces the randomness of the model to prevent hallucinated spikes.
* `top_p=0.9`: Prevents the model from exploring extreme outlier probabilities.

In [ ]:
#install if necessary:!pip install git+https://github.com/amazon-science/chronos-forecasting.git
from chronos import ChronosPipeline

# --- 1. MODEL INITIALIZATION ---
# We load the weights from Hugging Face directly onto the GPU.
# bfloat16 reduces memory usage by 50% without losing precision.
print("[INFO] Loading Foundation Model to GPU...")
pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-base",
    device_map="cuda" if torch.cuda.is_available() else "cpu",
    torch_dtype=torch.bfloat16,
)

# --- 2. GENERATION ---
# We pass our stacked tensor into the model. It looks at the prices/events and predicts the sales.
print("[INFO] Generating Probabilistic Forecast Paths...")
forecast = pipeline.predict(
    context_tensor,
    prediction_length=PREDICTION_LENGTH,
    num_samples=50,
    temperature=0.8,
    top_p=0.9
)

# --- 3. QUANTILE EXTRACTION ---
# The model outputs 50 guesses. Index 0 is our Target (Sales).
# We mathematically extract the Floor (10%), Median (50%), and Ceiling (90%) risk levels.
target_forecast = forecast[0].numpy()
low, median, high = np.quantile(target_forecast, [0.1, 0.5, 0.9], axis=0)
print("[SUCCESS] P10-P90 Risk Envelope successfully generated.")

[INFO] Loading Foundation Model to GPU...
[INFO] Generating Probabilistic Forecast Paths...
[SUCCESS] P10-P90 Risk Envelope successfully generated.


### Phase 6: Calibration Evaluation (wQL)

Traditional metrics like Mean Absolute Error (MAE) are useless here—predicting a flat "zero" often yields the lowest MAE on sparse data, which guarantees stockouts for a retailer.

Instead, we use **Mean Weighted Quantile Loss (wQL)**. This metric evaluates the entire "Cone of Uncertainty." It heavily penalizes the model if actual sales fall outside the P10-P90 risk envelope, ensuring our model is properly calibrated for safety stock planning. A score under 1.0 is considered highly viable.

In [ ]:
print("[INFO] Calculating Weighted Quantile Loss (wQL)...")

# --- 1. DEFINING THE TRUTH ---
# We use the LAST 28 days of our dataset as the "Ground Truth" to test the AI's guesses against.
actual_truth = item_df['sales'].values[-PREDICTION_LENGTH:]

# --- 2. WQL MATH FUNCTION ---
def calculate_wql(actual, forecast_quantile, quantile_level):
    # Calculates the Asymmetric Pinball Loss. It punishes the model more if it under-predicts a spike.
    diff = actual - forecast_quantile
    loss = np.maximum(quantile_level * diff, (quantile_level - 1) * diff)

    # Normalizes the score based on the total volume of sales.
    denominator = np.sum(np.abs(actual))
    return 2 * np.sum(loss) / denominator if denominator > 0 else 0

# --- 3. SCORING THE RISK ENVELOPE ---
# We score the Floor, the Median, and the Ceiling independently.
wql_p10 = calculate_wql(actual_truth, low, 0.10)
wql_p50 = calculate_wql(actual_truth, median, 0.50)
wql_p90 = calculate_wql(actual_truth, high, 0.90)

# We average them together for our final resume metric.
mean_wql = (wql_p10 + wql_p50 + wql_p90) / 3

# --- 4. OUTPUT ---
print(f"P10 wQL (Stockout Risk): {wql_p10:.4f}")
print(f"P50 wQL (Median Error):  {wql_p50:.4f}")
print(f"P90 wQL (Overstock Risk):{wql_p90:.4f}")
print("-" * 30)
print(f"Mean wQL Score:          {mean_wql:.4f}")

[INFO] Calculating Weighted Quantile Loss (wQL)...
P10 wQL (Stockout Risk): 0.6651
P50 wQL (Median Error):  0.9488
P90 wQL (Overstock Risk):0.8434
------------------------------
Mean wQL Score:          0.8191
